# F2 painterly layer rules v0.1

Fixed spring composition: **A** terrain, **B** terrain + schematic foreground vegetation, **C** the same candidate vegetation + density omission. Mist is a painterly reserve, not weather or white pigment.

Install the project once in your VSCode/Jupyter environment: `python -m pip install -e ".[research,dev]"` from the repository root. Select that environment as the notebook kernel. Keep the original MASTER/VEG TIFFs outside Git. This notebook never writes them.

In [ ]:
from pathlib import Path
import os, json
import numpy as np
from IPython.display import display, Image
from datascape.rendering.painterly_layers import run

ROOT = Path.cwd()
if not (ROOT / "configs").exists():
    ROOT = ROOT.parent
assert (ROOT / "configs/scenes/seoul_f2_painterly_layers_v01.json").is_file()
DATA = Path(os.environ.get("DATASCAPE_DATA_DIR", ROOT / "data/raw"))
MASTER = DATA / "SeoulNorth_v02_MASTER_30m.tif"
VEG = DATA / "SeoulNorth_v02_VEG_10m.tif"
assert MASTER.is_file() and VEG.is_file(), "Set DATASCAPE_DATA_DIR to the original TIFF folder."
CONFIG = ROOT / "configs/scenes/seoul_f2_painterly_layers_v01.json"
OUTPUT = ROOT / "outputs/f2_painterly_layers_v01"


In [ ]:
result = run(MASTER, VEG, CONFIG, OUTPUT, repo_root=ROOT)
display(Image(filename=str(OUTPUT / "ABC_comparison.png")))

## Inspect rules separately

M: valley/low-ground + depth + screen resting zone, with peak protection. W: stronger mist cores plus a data-weighted foreground passage. V: native seasonal NDVI/support × terrain factors × foreground role × composition balance × seeded density variation. Retention fields suppress terrain and vegetation separately. The corridor is a composition reserve, not an observed road or river.

In [ ]:
display(Image(filename=str(OUTPUT / "fields.png")))
display(Image(filename=str(OUTPUT / "data_and_omission.png")))
display(Image(filename=str(OUTPUT / "C_physical_pair.png")))

In [ ]:
f = result["fields"]
assert result["record"]["metrics"]["baseline_pixels_identical"]
assert np.all(result["images"]["C"][f["reserve"]] == 1)
assert result["record"]["inputs"]["master"]["unchanged_after"]
assert result["record"]["inputs"]["vegetation"]["unchanged_after"]
print(json.dumps(result["record"]["metrics"], indent=2))
print("Archives:", OUTPUT / "fields.npz", OUTPUT / "plants.json", OUTPUT / "record.json")

## Controlled variations

Copy the scene JSON to a new file and change one field parameter at a time, then run to a **new output directory**. Keep the F2 source configuration, terrain and source TIFFs fixed. `seed` affects density noise, common root placement/thinning and generic branching geometry through separate streams. B/C share the root set; C selects a subset and attenuates mark contributions.

This preview has no terrain stroke engine, species reconstruction, coloring, G-code or hardware execution. The field archive is the contract for later line and color omission. Schematic polylines are exploratory marks, not machine-ready brush strokes.